# Component 03 — Notebook 02: Train/Validation/Test Split and Normalization
**Research:** J26-IT-359 — Sri Lankan Endemic Bird Occurrence Prediction

**Input:** `data/processed/occurrence/targets_v1/preprocessing/bird_occurrence_cleaned_labelled.csv`
from Notebook 01. Its verified Y is `target_occurrence`: 1 = detected;
0 = not detected on an eligible complete survey, not confirmed biological absence.

Run top to bottom. The default is a **joint model for all 24 species**, including species
identity and survey effort as predictors. Set `SELECT_SPECIES` for a separate species model.
Spatial splitting is the default; random grouped and chronological temporal splits are also available.
Preprocessing is fitted on training data only. Validation/test data only use `transform`.

The target is survey detection within the selected-checklist sampling frame, not national occupancy.
Use the same saved splits for model comparisons and do not tune using the final test set.

**Normalization choice:** only `temperature_mean_c`, `rainfall_mm` and `elevation_m`
are normalized. Coordinates, NDVI, year, cyclical month features and any survey-effort
predictors retain their original units/ranges. Categorical variables are one-hot encoded.
The feature selection below was checked against the proposal's methodology and Table 4,
pages 16–18 of [Bird occurence pattern prediction com -03.pdf](<docs/Bird occurence pattern prediction com -03.pdf>).


## Cell 1 — Import libraries and set paths

In [21]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder

RANDOM_STATE = 42
TARGET = 'target_occurrence'
SPLIT_METHOD = 'spatial'  # 'random', 'spatial', or 'temporal'
SELECT_SPECIES = None  # None = joint model; or enter a scientific name, e.g. 'Gallus lafayettii'.
INCLUDE_EFFORT_FEATURES = True  # Recommended for survey detection; disable for a defined environment-only experiment.
SCALING = 'minmax'  # Applied ONLY to temperature, rainfall and elevation; optionally 'standard'.

DATA_RELATIVE_PATH = Path('data/processed/occurrence/targets_v1/preprocessing/bird_occurrence_cleaned_labelled.csv')
PROJECT_ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / DATA_RELATIVE_PATH).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Cannot find {DATA_RELATIVE_PATH}. Run Notebook 01 from the occurrence_prediction component first.')
DATA_PATH = PROJECT_ROOT / DATA_RELATIVE_PATH
species_scope_name = 'joint' if SELECT_SPECIES is None else SELECT_SPECIES.replace(' ', '_')
experiment_name = f'{species_scope_name}_{SCALING}_seed{RANDOM_STATE}'
experiment_name += '_effort' if INCLUDE_EFFORT_FEATURES else '_environment'
RUN_ID = datetime.now().strftime('run_%Y%m%d_%H%M%S_%f')
OUTPUT_DIR = PROJECT_ROOT / 'data/splits/occurrence/targets_v1' / SPLIT_METHOD / experiment_name / RUN_ID
# The save cell creates this directory after the split and preprocessing checks pass.
print('Reading:', DATA_PATH)
print('Split method:', SPLIT_METHOD, '| Species selection:', SELECT_SPECIES or 'all species')
print('Output directory:', OUTPUT_DIR)

Reading: /Users/keerthikeerthi/Downloads/component3/data/processed/occurrence/targets_v1/preprocessing/bird_occurrence_cleaned_labelled.csv
Split method: spatial | Species selection: all species
Output directory: /Users/keerthikeerthi/Downloads/component3/data/splits/occurrence/targets_v1/spatial/joint_minmax_seed42_effort


## Cell 2 — Read the previous notebook output and understand the data

In [22]:
df = pd.read_csv(DATA_PATH, low_memory=False)
required = {'checklist_id', 'survey_group_id', 'scientific_name', 'cell_id',
            'observation_date', 'year', 'month', 'latitude', 'longitude', TARGET}
missing_columns = required - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
print('Dataset shape:', df.shape)
print('Species:', df['scientific_name'].nunique(), '| Surveys:', df['survey_group_id'].nunique())
print('Column names:', df.columns.tolist())
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(20))

Dataset shape: (505608, 39)
Species: 24 | Surveys: 21067
Column names: ['checklist_id', 'survey_group_id', 'scientific_name', 'observation_date', 'year', 'month', 'cell_id', 'latitude', 'longitude', 'temperature_mean_c', 'rainfall_mm', 'elevation_m', 'ndvi', 'land_cover', 'duration_minutes', 'effort_distance_km', 'survey_distance_km', 'protocol', 'all_species_reported', 'hours_of_day', 'number_observers', 'effort_speed_kmph', 'target_occurrence', 'label_status', 'invalid_coordinates', 'invalid_month', 'date_mismatch', 'invalid_ndvi', 'invalid_rainfall', 'invalid_duration', 'invalid_distance', 'invalid_target', 'month_sin', 'month_cos', 'ndvi_missing', 'elevation_m_missing', 'duration_minutes_missing', 'survey_distance_km_missing', 'cell_id_missing']


,checklist_id,survey_group_id,scientific_name,observation_date,year,month,cell_id,latitude,longitude,temperature_mean_c,...,invalid_duration,invalid_distance,invalid_target,month_sin,month_cos,ndvi_missing,elevation_m_missing,duration_minutes_missing,survey_distance_km_missing,cell_id_missing
0,S100000513,S100000513,Argya cinereifrons,2022-01-01,2022,1,lka_laea_5000m_1_-15,7.22606,80.75056,22.6,...,False,False,False,0.5,0.866025,0,0,0,0,0
1,S100000513,S100000513,Centropus chlororhynchos,2022-01-01,2022,1,lka_laea_5000m_1_-15,7.22606,80.75056,22.6,...,False,False,False,0.5,0.866025,0,0,0,0,0
2,S100000513,S100000513,Chrysocolaptes stricklandi,2022-01-01,2022,1,lka_laea_5000m_1_-15,7.22606,80.75056,22.6,...,False,False,False,0.5,0.866025,0,0,0,0,0
3,S100000513,S100000513,Columba torringtoniae,2022-01-01,2022,1,lka_laea_5000m_1_-15,7.22606,80.75056,22.6,...,False,False,False,0.5,0.866025,0,0,0,0,0
4,S100000513,S100000513,Dicrurus lophorinus,2022-01-01,2022,1,lka_laea_5000m_1_-15,7.22606,80.75056,22.6,...,False,False,False,0.5,0.866025,0,0,0,0,0


effort_distance_km            218376
ndvi                          183360
elevation_m                      888
land_cover                        48
checklist_id                       0
invalid_coordinates                0
invalid_month                      0
date_mismatch                      0
invalid_ndvi                       0
invalid_rainfall                   0
invalid_duration                   0
target_occurrence                  0
invalid_distance                   0
invalid_target                     0
month_sin                          0
month_cos                          0
ndvi_missing                       0
elevation_m_missing                0
duration_minutes_missing           0
survey_distance_km_missing         0
dtype: int64

## Cell 3 — Verify Y before continuing
This file retains the previously verified detection/non-detection labels. No new zeros
are created here. Y must be binary and non-missing; label provenance stays outside X.

In [23]:
print('Target counts (including missing):')
print(df[TARGET].value_counts(dropna=False))
if df[TARGET].isna().any() or not df[TARGET].isin([0, 1]).all():
    raise ValueError('Y must contain verified 0/1 labels only. Do not impute missing labels.')
if df[TARGET].nunique() != 2:
    raise ValueError('Both target classes are required for binary classification.')
df[TARGET] = df[TARGET].astype('int8')
if 'label_status' in df.columns:
    expected_status = df[TARGET].map({0: 'verified_non_detection', 1: 'detected'})
    if not df['label_status'].eq(expected_status).all():
        raise ValueError('Some Y values disagree with their verified label status.')
    print('Label status agrees with Y and will be excluded from X.')
display(df[TARGET].value_counts(normalize=True).rename('fraction'))

Target counts (including missing):
target_occurrence
0    451395
1     54213
Name: count, dtype: int64
Label status agrees with Y and will be excluded from X.


target_occurrence
0    0.892777
1    0.107223
Name: fraction, dtype: float64

## Cell 4 — Basic data checks (without filling numeric missing values)
**Why not fill now?** We must calculate imputation medians using training data only. The split happens first.

In [24]:
print('Exact duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().copy()
if df.duplicated(['checklist_id', 'scientific_name']).any():
    raise ValueError('Conflicting checklist/species rows need review before splitting.')
if df[['checklist_id', 'survey_group_id', 'scientific_name', 'cell_id', 'observation_date']].isna().any().any():
    raise ValueError('Survey identifiers, species, grid cells and dates must be valid for splitting.')

for column in ['latitude', 'longitude', 'year', 'month', 'temperature_mean_c',
               'rainfall_mm', 'elevation_m', 'ndvi', 'duration_minutes',
               'survey_distance_km', 'number_observers', 'hours_of_day', 'effort_speed_kmph']:
    if column in df:
        df[column] = pd.to_numeric(df[column], errors='coerce')
if not (df['latitude'].between(-90, 90).all() and df['longitude'].between(-180, 180).all()):
    raise ValueError('Invalid coordinates require review.')
if not (df['year'].eq(df['observation_date'].dt.year).all() and
        df['month'].eq(df['observation_date'].dt.month).all()):
    raise ValueError('Date, year and month disagree.')
if not df['month'].between(1, 12).all():
    raise ValueError('Invalid month values require review.')
if (df['ndvi'].notna() & ~df['ndvi'].between(-0.2, 1)).any():
    raise ValueError('Invalid NDVI values require review before normalization.')
if (df['rainfall_mm'].notna() & df['rainfall_mm'].lt(0)).any():
    raise ValueError('Negative rainfall values require review.')

# Every survey and its species rows must agree on split-defining fields.
group_consistency = df.groupby('survey_group_id')[['checklist_id', 'cell_id', 'year', 'observation_date']].nunique(dropna=False)
if group_consistency.gt(1).any().any():
    raise ValueError('A survey spans conflicting checklists/cells/dates; resolve it before splitting.')
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df = df.reset_index(drop=True)
df['_row_id'] = np.arange(len(df))
print('Rows after checks:', len(df), '| No numeric missing values have been imputed.')

Exact duplicate rows: 0
Rows after checks: 505608 | No numeric missing values have been imputed.


## Cell 5 — X features checked against the research proposal
Sections 3.1–3.2 and Table 4 (pages 16–18) specify temperature, rainfall,
elevation, vegetation, land cover, coordinates, derived time features and available
checklist/effort information. Sections 3.3 and 3.5 require the same core features
and validation partitions for Random Forest and XGBoost.

| Input to X | Preparation | Research purpose |
| --- | --- | --- |
| `temperature_mean_c`, `rainfall_mm`, `elevation_m` | Training-only median imputation, then MinMax normalization | Weather and topographic conditions |
| `latitude`, `longitude` | Unscaled | Geographic location |
| `ndvi` | Training-only imputation for gaps; observed values remain unscaled | Vegetation condition |
| `year` | Unscaled | Temporal context |
| `month_sin`, `month_cos` | Unscaled; keep their original cyclical ranges | Seasonal timing |
| `land_cover` | One-hot encoding | Land-cover class |
| `scientific_name` | One-hot encoding for the joint model | Identify the target species |
| Optional duration, survey distance, observer count and start time | Unscaled | Observation effort/detectability controls |
| Optional `protocol` | One-hot encoding | Observation protocol |

Raw `month` is replaced in X by its sine/cosine representation. `cell_id`, dates,
survey IDs and label-provenance fields remain available for splitting/evaluation,
and are excluded from X. Y remains `target_occurrence`, with no normalization.

`SELECT_SPECIES = None` retains all 24 species. A single-species experiment drops
the constant species-name predictor. `INCLUDE_EFFORT_FEATURES` controls the optional
survey variables listed in Table 4; define an effort scenario when using them for
candidate-area maps. The present labels describe survey detection.

The proposal does not prescribe these three normalization columns. This notebook
uses your requested preprocessing choice. Missingness indicators are derived
using training data and remain binary and unscaled.

In [25]:
if SELECT_SPECIES is not None:
    if SELECT_SPECIES not in set(df['scientific_name']):
        raise ValueError(f'Selected species is not present: {SELECT_SPECIES}')
    df = df[df['scientific_name'].eq(SELECT_SPECIES)].copy()
    print('Selected species:', SELECT_SPECIES, '| Rows:', len(df))

# Only these three columns are passed to a scaler.
normalization_features = ['temperature_mean_c', 'rainfall_mm', 'elevation_m']
unscaled_numeric_features = ['latitude', 'longitude', 'ndvi', 'year', 'month_sin', 'month_cos']
categorical_features = ['land_cover']
if df['scientific_name'].nunique() > 1:
    categorical_features.insert(0, 'scientific_name')
if INCLUDE_EFFORT_FEATURES:
    unscaled_numeric_features += ['duration_minutes', 'survey_distance_km', 'number_observers', 'hours_of_day']
    categorical_features += ['protocol']

numeric_features = normalization_features + unscaled_numeric_features
feature_columns = numeric_features + categorical_features
if set(feature_columns) - set(df.columns):
    raise ValueError(f'Missing predictors: {sorted(set(feature_columns) - set(df.columns))}')
forbidden_features = {TARGET, 'label_status', 'record_id', 'checklist_id', 'survey_group_id',
                      'cell_id', 'all_species_reported', '_row_id', 'month'}
assert forbidden_features.isdisjoint(feature_columns)
assert set(normalization_features).isdisjoint(unscaled_numeric_features)
assert len(feature_columns) == len(set(feature_columns))
print('Normalize only:', normalization_features)
print('Numeric X kept unscaled:', unscaled_numeric_features)
print('One-hot encode:', categorical_features)
print('Y (not normalized):', TARGET)
display(pd.DataFrame({
    'feature': feature_columns,
    'preparation': (['median imputation + normalization'] * len(normalization_features)
                    + ['median imputation only; no scaling'] * len(unscaled_numeric_features)
                    + ['one-hot encoding; no numeric scaling'] * len(categorical_features))
}))

Normalize only: ['temperature_mean_c', 'rainfall_mm', 'elevation_m']
Numeric X kept unscaled: ['latitude', 'longitude', 'ndvi', 'year', 'month_sin', 'month_cos', 'duration_minutes', 'survey_distance_km', 'number_observers', 'hours_of_day']
One-hot encode: ['scientific_name', 'land_cover', 'protocol']
Y (not normalized): target_occurrence


,feature,preparation
0,temperature_mean_c,median imputation + normalization
1,rainfall_mm,median imputation + normalization
2,elevation_m,median imputation + normalization
3,latitude,median imputation only; no scaling
4,longitude,median imputation only; no scaling
5,ndvi,median imputation only; no scaling
6,year,median imputation only; no scaling
7,month_sin,median imputation only; no scaling
8,month_cos,median imputation only; no scaling
9,duration_minutes,median imputation only; no scaling


## Cell 6 — Create train (about 80%), validation (about 10%) and test (about 10%)
**Random:** group by `survey_group_id`, never by individual species rows.
**Spatial:** hold out whole `cell_id` groups and also verify survey separation.
**Temporal:** train on earlier years, validate/test on later years.
Group splits are approximate by rows, and neighbouring cells may still be correlated.
The split report checks both global and per-species class coverage.

**Checked on the current dataset:** the default spatial split (seed 42) has no validation detections for
`Argya cinereifrons`, `Otus thilohoffmanni`, `Phaenicophaeus pyrrhocephalus`,
`Sturnornis albofrontatus` and `Zoothera imbricata`. All species have both classes in training
and test. The coverage report flags these validation cases; per-species ROC-AUC is undefined
when a partition has only one class. Review the evaluation design before comparing models.


In [26]:
if SPLIT_METHOD not in {'random', 'spatial', 'temporal'}:
    raise ValueError('SPLIT_METHOD must be random, spatial, or temporal.')
if SPLIT_METHOD == 'temporal':
    years = sorted(df['year'].unique())
    if len(years) < 4:
        raise ValueError('Need at least four distinct years for this temporal split.')
    a = max(1, int(len(years) * 0.8))
    b = min(max(a + 1, int(len(years) * 0.9)), len(years) - 1)
    train_df = df[df['year'].isin(years[:a])].copy()
    val_df = df[df['year'].isin(years[a:b])].copy()
    test_df = df[df['year'].isin(years[b:])].copy()
    assert train_df['year'].max() < val_df['year'].min()
    assert val_df['year'].max() < test_df['year'].min()
    print('Train years:', years[:a], '| Validation years:', years[a:b], '| Test years:', years[b:])
else:
    group_col = 'cell_id' if SPLIT_METHOD == 'spatial' else 'survey_group_id'
    if df[group_col].nunique() < 5:
        raise ValueError('Not enough distinct groups for the requested three-way split.')
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    train_i, remaining_i = next(splitter.split(df, groups=df[group_col]))
    train_df = df.iloc[train_i].copy()
    remaining = df.iloc[remaining_i]
    splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
    val_i, test_i = next(splitter2.split(remaining, groups=remaining[group_col]))
    val_df = remaining.iloc[val_i].copy()
    test_df = remaining.iloc[test_i].copy()
    for first, second in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
        assert set(first[group_col]).isdisjoint(second[group_col])

partitions = {'train': train_df, 'validation': val_df, 'test': test_df}
split_summary = []
species_class_reports = []
for name, frame in partitions.items():
    if frame.empty or frame[TARGET].nunique() != 2:
        raise ValueError(f'{name} is empty or contains only one global target class.')
    split_summary.append({'split': name, 'rows': len(frame), 'row_percent': len(frame) / len(df) * 100,
                          'surveys': frame['survey_group_id'].nunique(), 'cells': frame['cell_id'].nunique(),
                          'detected_y1': int(frame[TARGET].sum()), 'not_detected_y0': int(frame[TARGET].eq(0).sum()),
                          'first_year': int(frame['year'].min()), 'last_year': int(frame['year'].max())})
    counts = pd.crosstab(frame['scientific_name'], frame[TARGET]).reindex(
        index=sorted(df['scientific_name'].unique()), columns=[0, 1], fill_value=0)
    counts.columns = ['not_detected_y0', 'detected_y1']
    counts['split'] = name
    counts['both_classes'] = counts['not_detected_y0'].gt(0) & counts['detected_y1'].gt(0)
    species_class_reports.append(counts.reset_index())

for first, second in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert set(first['_row_id']).isdisjoint(second['_row_id'])
    assert set(first['survey_group_id']).isdisjoint(second['survey_group_id'])
    assert set(first['checklist_id']).isdisjoint(second['checklist_id'])
assert set().union(*(set(frame['_row_id']) for frame in partitions.values())) == set(df['_row_id'])
assert sum(len(frame) for frame in partitions.values()) == len(df)
split_summary = pd.DataFrame(split_summary)
species_class_report = pd.concat(species_class_reports, ignore_index=True)
display(split_summary)
display(species_class_report)
one_class_species = species_class_report[~species_class_report['both_classes']]
if not one_class_species.empty:
    print('Some species lack one class in a partition; per-species AUC is undefined there. Review this coverage:')
    display(one_class_species)
print('Verified full row coverage and no row, checklist or survey overlap.')

,split,rows,row_percent,surveys,cells,detected_y1,not_detected_y0,first_year,last_year
0,train,416448,82.365785,17352,763,46021,370427,2010,2023
1,validation,51648,10.215028,2152,95,4578,47070,2010,2023
2,test,37512,7.419186,1563,96,3614,33898,2010,2023


,scientific_name,not_detected_y0,detected_y1,split,both_classes
0,Argya cinereifrons,16907,445,train,True
1,Centropus chlororhynchos,16831,521,train,True
2,Chrysocolaptes stricklandi,15982,1370,train,True
3,Columba torringtoniae,16602,750,train,True
4,Dicrurus lophorinus,16363,989,train,True
...,...,...,...,...,...
67,Sturnornis albofrontatus,1549,14,test,True
68,Tephrodornis affinis,1462,101,test,True
69,Treron pompadora,1378,185,test,True
70,Urocissa ornata,1551,12,test,True


Some species lack one class in a partition; per-species AUC is undefined there. Review this coverage:


,scientific_name,not_detected_y0,detected_y1,split,both_classes
24,Argya cinereifrons,2152,0,validation,False
36,Otus thilohoffmanni,2152,0,validation,False
38,Phaenicophaeus pyrrhocephalus,2152,0,validation,False
43,Sturnornis albofrontatus,2152,0,validation,False
47,Zoothera imbricata,2152,0,validation,False


Verified full row coverage and no row, checklist or survey overlap.
